In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

import mlflow

In [3]:
path_raw = "../data/processed/hrss_clean_integrated_original_columns.csv"
path_eng = "../data/processed/hrss_clean_integrated_with_engineered_features.csv"

df_raw = pd.read_csv(path_raw)
df_eng = pd.read_csv(path_eng)

print(df_raw.shape)
print(df_eng.shape)

(46898, 21)
(46898, 27)


In [4]:
print("RAW DATA INFO")
display(df_raw.info())

print("\nENGINEERED DATA INFO")
display(df_eng.info())

RAW DATA INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46898 entries, 0 to 46897
Data columns (total 21 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Timestamp        46898 non-null  float64
 1   I_w_BLO_Weg      46898 non-null  int64  
 2   O_w_BLO_power    46898 non-null  int64  
 3   O_w_BLO_voltage  46898 non-null  int64  
 4   I_w_BHL_Weg      46898 non-null  int64  
 5   O_w_BHL_power    46898 non-null  int64  
 6   O_w_BHL_voltage  46898 non-null  int64  
 7   I_w_BHR_Weg      46898 non-null  int64  
 8   O_w_BHR_power    46898 non-null  int64  
 9   O_w_BHR_voltage  46898 non-null  int64  
 10  I_w_BRU_Weg      46898 non-null  int64  
 11  O_w_BRU_power    46898 non-null  int64  
 12  O_w_BRU_voltage  46898 non-null  int64  
 13  I_w_HR_Weg       46898 non-null  int64  
 14  O_w_HR_power     46898 non-null  int64  
 15  O_w_HR_voltage   46898 non-null  int64  
 16  I_w_HL_Weg       46898 non-null  int64  
 17

None


ENGINEERED DATA INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46898 entries, 0 to 46897
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Timestamp               46898 non-null  float64
 1   I_w_BLO_Weg             46898 non-null  int64  
 2   O_w_BLO_power           46898 non-null  int64  
 3   O_w_BLO_voltage         46898 non-null  int64  
 4   I_w_BHL_Weg             46898 non-null  int64  
 5   O_w_BHL_power           46898 non-null  int64  
 6   O_w_BHL_voltage         46898 non-null  int64  
 7   I_w_BHR_Weg             46898 non-null  int64  
 8   O_w_BHR_power           46898 non-null  int64  
 9   O_w_BHR_voltage         46898 non-null  int64  
 10  I_w_BRU_Weg             46898 non-null  int64  
 11  O_w_BRU_power           46898 non-null  int64  
 12  O_w_BRU_voltage         46898 non-null  int64  
 13  I_w_HR_Weg              46898 non-null  int64  
 14  O_w_HR_power    

None

In [5]:
print("RAW TARGET DISTRIBUTION")
print(df_raw['operation_type'].value_counts(normalize=True))

print("\nENGINEERED TARGET DISTRIBUTION")
print(df_eng['operation_type'].value_counts(normalize=True))

RAW TARGET DISTRIBUTION
operation_type
0    0.54851
1    0.45149
Name: proportion, dtype: float64

ENGINEERED TARGET DISTRIBUTION
operation_type
0    0.54851
1    0.45149
Name: proportion, dtype: float64


In [6]:
target = "operation_type"

X_raw = df_raw.drop(columns=[target])
y_raw = df_raw[target]

X_eng = df_eng.drop(columns=[target])
y_eng = df_eng[target]

print(X_raw.shape, y_raw.shape)
print(X_eng.shape, y_eng.shape)

(46898, 20) (46898,)
(46898, 26) (46898,)


In [7]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y_raw,
    test_size=0.2,
    random_state=42,
    stratify=y_raw
)

X_train_eng, X_test_eng, y_train_eng, y_test_eng = train_test_split(
    X_eng, y_eng,
    test_size=0.2,
    random_state=42,
    stratify=y_eng
)

print(X_train_raw.shape, X_test_raw.shape)
print(X_train_eng.shape, X_test_eng.shape)

(37518, 20) (9380, 20)
(37518, 26) (9380, 26)


In [8]:
import os

os.makedirs("../data/splits", exist_ok=True)

# RAW
X_train_raw.to_csv("../data/splits/X_train_raw.csv", index=False)
X_test_raw.to_csv("../data/splits/X_test_raw.csv", index=False)
y_train_raw.to_csv("../data/splits/y_train_raw.csv", index=False)
y_test_raw.to_csv("../data/splits/y_test_raw.csv", index=False)

# ENGINEERED
X_train_eng.to_csv("../data/splits/X_train_eng.csv", index=False)
X_test_eng.to_csv("../data/splits/X_test_eng.csv", index=False)
y_train_eng.to_csv("../data/splits/y_train_eng.csv", index=False)
y_test_eng.to_csv("../data/splits/y_test_eng.csv", index=False)

In [9]:
mlflow.set_tracking_uri("file:../mlruns")

mlflow.set_experiment("HRSS_Operational_Classification")

<Experiment: artifact_location='file:///d:/ARFI/Kuliah/Project/industrial_recommendation_system/notebooks/../mlruns/691691579525109979', creation_time=1779957094958, experiment_id='691691579525109979', last_update_time=1779957094958, lifecycle_stage='active', name='HRSS_Operational_Classification', tags={}>

In [10]:
experiment_config = {
    "target": "operation_type",
    "problem_type": "binary_classification",
    "domain": "industrial_telemetry_hrss",
    "objective": "operational_efficiency_classification",
    "primary_metric": "f1_macro",
    "secondary_metric": "recall_class_1"
}

print(experiment_config)

{'target': 'operation_type', 'problem_type': 'binary_classification', 'domain': 'industrial_telemetry_hrss', 'objective': 'operational_efficiency_classification', 'primary_metric': 'f1_macro', 'secondary_metric': 'recall_class_1'}


In [11]:
experiment_registry = {
    "raw": {
        "description": "Baseline model using original telemetry features only",
        "dataset_shape": X_train_raw.shape
    },
    "engineered": {
        "description": "Enhanced model using engineered operational features",
        "dataset_shape": X_train_eng.shape
    }
}

print(experiment_registry)

{'raw': {'description': 'Baseline model using original telemetry features only', 'dataset_shape': (37518, 20)}, 'engineered': {'description': 'Enhanced model using engineered operational features', 'dataset_shape': (37518, 26)}}


In [12]:
print("READY FOR EXPERIMENTS")

print("\nRAW:")
print(X_train_raw.shape, X_test_raw.shape)

print("\nENGINEERED:")
print(X_train_eng.shape, X_test_eng.shape)

READY FOR EXPERIMENTS

RAW:
(37518, 20) (9380, 20)

ENGINEERED:
(37518, 26) (9380, 26)
